In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
    
GOOGLE_MODEL = "gemini-3.1-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=GOOGLE_MODEL, 
    temperature=0,
)

In [3]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

response_schemas = [
    ResponseSchema(name="destination", description="여행지"),
    ResponseSchema(name="duration", description="여행 기간 (예: 2박 3일)"),
    ResponseSchema(name="budget", description="여행에 사용한 예산 (예: 30만원)"),
    ResponseSchema(name="rating", description="추천도 (1~5 사이의 숫자만)"),
    ResponseSchema(name="activities", description="주요 활동 목록", type="list"),
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = parser.get_format_instructions()

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 후기나 계획 텍스트에서 핵심 정보를 추출하는 AI입니다. 글에 있는 내용만 사용해 정보를 추출하세요.\n{format_instructions}"),
    ("user", "여행 글: {travel_text}"),
]).partial(format_instructions=format_instructions)

chain = prompt | llm | parser

In [7]:
import json

travel_text =  "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."

result = chain.invoke({"travel_text": travel_text})

print(json.dumps(result, indent=4, ensure_ascii=False))

{
    "destination": "부산",
    "duration": "2박 3일",
    "budget": "30만원",
    "rating": "4",
    "activities": [
        "해운대 바다 구경",
        "자갈치시장에서 회 먹기",
        "감천문화마을 구경"
    ]
}
